# Data Cleaning — Orders Table
**Source:** `olist_orders_dataset.csv`  
**Output:** `data/cleaned/orders_cleaned.csv`

### Cleaning Steps
1. Load raw data
2. Initial inspection (shape, columns, missing values, status distribution)
3. Cross-check missing values vs order status
4. Remove logically inconsistent rows
5. Convert date columns to datetime
6. Create derived columns
7. Validate derived columns
8. Export cleaned data

## Step 1 — Load Raw Data

In [14]:
import pandas as pd

# ---- 1. Load raw data ───────────────
df = pd.read_csv('../data/raw/olist_orders_dataset.csv')

## Step 2 — Initial Inspection

In [26]:
# ---- 2. Initial inspection ───────────────
print("=== shape ===")
print(df.shape)

print("\n=== Column names ===")
print(df.columns.tolist())

print("\n=== First 5 rows ===")
print(df.head())

print("\n=== Missing values ===")
print(df.isnull().sum())

# Numeric columns summary
print(df.describe())

# Categorical columns summary
print(df.describe(include='str'))

print("\n=== Order status distribution ===")
print(df['order_status'].value_counts())
print(df['order_status'].value_counts(normalize=True).round(4) * 100)

=== shape ===
(99418, 10)

=== Column names ===
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_delay_days', 'estimated_delivery_days']

=== First 5 rows ===
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp   order_approved_at  \
0    delivered      2017-10-02 10:56:33 2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37 2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49 2018-08-08 08:55:23  

## Step 3 — Cross-check Missing Values vs Order Status

Missing values in date columns are expected for orders that have not completed the full delivery flow.  
However, any `delivered` order missing a date column is logically inconsistent and should be removed.

In [16]:
# ── 3. Cross-check missing values vs order_status ─────────────
missing_cols = [
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date'
]

for col in missing_cols:
    print(f"\n=== Missing [{col}] by order_status ===")
    print(df[df[col].isnull()]['order_status'].value_counts())


=== Missing [order_approved_at] by order_status ===
order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

=== Missing [order_delivered_carrier_date] by order_status ===
order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

=== Missing [order_delivered_customer_date] by order_status ===
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


## Step 4 — Remove Logically Inconsistent Rows

A `delivered` order must have all three date columns populated.  
Rows where `order_status == delivered` but any date is missing are data errors and will be dropped.

In [17]:
# ── 4. Remove logically inconsistent delivered orders ─────────
# delivered orders missing approval timestamp
mask1 = (df['order_status'] == 'delivered') & (df['order_approved_at'].isnull())

# delivered orders missing carrier handover timestamp
mask2 = (df['order_status'] == 'delivered') & (df['order_delivered_carrier_date'].isnull())

# delivered orders missing customer delivery timestamp
mask3 = (df['order_status'] == 'delivered') & (df['order_delivered_customer_date'].isnull())

rows_before = len(df)
df = df[~(mask1 | mask2 | mask3)]
rows_after = len(df)

print(f"Rows removed: {rows_before - rows_after}")
print(f"Rows remaining: {rows_after}")

Rows removed: 23
Rows remaining: 99418


## Step 5 — Convert Date Columns to Datetime

In [18]:
# ── Check current dtypes before conversion ───────────────────
print("=== Current dtypes ===")
print(df.dtypes)

=== Current dtypes ===
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object


In [19]:
# ── 5. Convert date columns to datetime ──────────────────────
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col])

print("=== Dtypes after conversion ===")
print(df[date_cols].dtypes)

=== Dtypes after conversion ===
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


## Step 6 — Create Derived Columns

- `delivery_delay_days`: actual delivery date minus estimated delivery date. Positive = late, negative = early.
- `estimated_delivery_days`: days from purchase to estimated delivery date.

In [20]:
# ── 6. Create derived columns ─────────────────────────────────

# Actual delivery days vs estimated (positive = late, negative = early)
df['delivery_delay_days'] = (
    df['order_delivered_customer_date'] - df['order_estimated_delivery_date']
).dt.days

# Days from purchase to estimated delivery
df['estimated_delivery_days'] = (
    df['order_estimated_delivery_date'] - df['order_purchase_timestamp']
).dt.days

print("=== Derived columns preview ===")
print(df[['order_id', 'delivery_delay_days', 'estimated_delivery_days']].head(10))

print("\n=== delivery_delay_days stats ===")
print(df['delivery_delay_days'].describe())

=== Derived columns preview ===
                           order_id  delivery_delay_days  \
0  e481f51cbdc54678b7cc49136f2d6af7                 -8.0   
1  53cdb2fc8bc7dce0b6741e2150273451                 -6.0   
2  47770eb9100c2d0c44946d9cf07ec65d                -18.0   
3  949d5b44dbf5de918fe9c16f97b45f8a                -13.0   
4  ad21c59c0840e6cb83a9ceb5573f8159                -10.0   
5  a4591c265e18cb1dcee52889e2d8acc3                 -6.0   
6  136cce7faa42fdb2cefd53fdc79a6098                  NaN   
7  6514b8ad8028c9f2cc2374ded245783f                -12.0   
8  76c6e866289321a7c93b82b54852dc33                -32.0   
9  e69bfb5eb88e0ed6a785585b27e16dbf                 -7.0   

   estimated_delivery_days  
0                       15  
1                       19  
2                       26  
3                       26  
4                       12  
5                       22  
6                       27  
7                       21  
8                       41  
9                

## Step 7 — Validate Derived Columns

Check that NaN values in `delivery_delay_days` only appear in non-delivered orders (expected behaviour).  
Check for extreme values — retained as real business events, not data errors.

In [21]:
# ── 7a. Check NaN delay corresponds to non-delivered orders ───
print("=== Order status for rows with NaN delivery_delay_days ===")
print(df[df['delivery_delay_days'].isnull()]['order_status'].value_counts())

=== Order status for rows with NaN delivery_delay_days ===
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
created           5
approved          2
Name: count, dtype: int64


In [22]:
# ── 7b. Check extreme delivery delay values ───────────────────
print("=== Extremely early (< -30 days) ===")
print(len(df[df['delivery_delay_days'] < -30]))

print("\n=== Extremely late (> 30 days) ===")
print(len(df[df['delivery_delay_days'] > 30]))

=== Extremely early (< -30 days) ===
2354

=== Extremely late (> 30 days) ===
345


In [23]:
# ── 7c. Check for duplicate order_id ─────────────────────────
print(f"Total rows: {len(df)}")
print(f"Unique order_id: {df['order_id'].nunique()}")

Total rows: 99418
Unique order_id: 99418


## Step 8 — Export Cleaned Data

In [24]:
# ── 8. Export cleaned data ────────────────────────────────────
df.to_csv('../data/cleaned/orders_cleaned.csv', index=False)

print(f"Exported: {len(df)} rows")
print("Saved to: data/cleaned/orders_cleaned.csv")

Exported: 99418 rows
Saved to: data/cleaned/orders_cleaned.csv
